# Clean CNP Workflow (No `resum` Dependencies)

This notebook uses `cnp_clean_pipeline.py` for:
- training on H5 files
- prediction export to CSV (`y_cnp`, `y_cnp_err`, `y_raw`, `fidelity`, `iteration`)
- training and prediction plots


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

from cnp_clean_pipeline import load_runtime_config, train_cnp, predict_cnp


## Run Mixup First
If `use_data_augmentation: mixup` in `settings2.yaml`, run this once before training:
```bash
python src/run_cnp/preprocess_mixup_xenon2.py --config src/xenon/settings2.yaml
```


In [ ]:
# Update this path if needed
CONFIG_PATH = Path('../xenon/settings2.yaml').resolve()
runtime = load_runtime_config(CONFIG_PATH, seed=42)

print('config      :', runtime.config_path)
print('train_dir   :', runtime.train_dir)
print('predict_dirs:', runtime.predict_dirs)
print('out_dir     :', runtime.out_dir)
print('epochs      :', runtime.epochs)
print('plot_after  :', runtime.plot_after)


## Train

`steps_per_epoch` controls runtime. Increase on lab PC for full training.


In [ ]:
train_result = train_cnp(
    runtime,
    steps_per_epoch=5000,
    lr=1e-4,
    repr_dim=32,
    hidden=128,
    dropout=0.1,
    monitor_every=1000,
    show_monitor_plots=True,
)
train_result


In [ ]:
display(Image(filename=str(train_result.history_plot)))
display(Image(filename=str(train_result.sample_plot)))


## Predict

By default, suffix is inferred from `path_to_files_predict` (`output` or `output_validation`).


In [ ]:
predict_result = predict_cnp(
    runtime,
    model_path=train_result.model_path,
    mc_samples=30,
    chunk_size=20000,
)
predict_result


In [ ]:
df = pd.read_csv(predict_result.csv_path)
print(df.shape)
df.head()


In [ ]:
display(Image(filename=str(predict_result.heatmap_path)))
display(Image(filename=str(predict_result.error_heatmap_path)))


## CLI Equivalent

```bash
python src/run_cnp/cnp_clean_pipeline.py --config src/xenon/settings2.yaml train --steps-per-epoch 500
python src/run_cnp/cnp_clean_pipeline.py --config src/xenon/settings2.yaml predict --model-path src/xenon/out/cnp/cnp_v101.0_model_15epochs.pth
```
